# 05 — Export Models & Embeddings

Prepare trained models for deployment in the FastAPI sidecar:
1. Export semantic model checkpoint
2. Re-embed all anime with the fine-tuned model → 384-dim vectors
3. Export CF model checkpoint + anime index
4. Package everything into the `ml-models/` directory

**Requires**: `03_semantic_training.ipynb` and `04_cf_training.ipynb` completed.

In [1]:
!pip install -q sentence-transformers torch scipy tqdm

In [2]:
import json
import shutil
import numpy as np
import torch
from tqdm.auto import tqdm
from pathlib import Path
from sentence_transformers import SentenceTransformer
from datetime import datetime, timezone

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
# Prefer project-root ml-models/ when running from notebooks/ locally.
EXPORT_DIR = Path("../ml-models") if Path("../docker-compose.yml").exists() else Path("ml-models")
EXPORT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
EXPORT_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


Device: cuda


## 1. Export Semantic Model

In [3]:
# Copy the fine-tuned sentence transformer to export directory
semantic_src = MODEL_DIR / "anime_semantic_multipos"
semantic_dst = EXPORT_DIR / "semantic"

if semantic_dst.exists():
    shutil.rmtree(semantic_dst)
shutil.copytree(semantic_src, semantic_dst)

# Verify it loads
model = SentenceTransformer(str(semantic_dst), device=str(device))
test_embed = model.encode(["test sentence"])
print(f"Semantic model exported: {semantic_dst}")
print(f"  Embedding dim: {test_embed.shape[1]}")

# Calculate model size
total_size = sum(f.stat().st_size for f in semantic_dst.rglob("*") if f.is_file())
print(f"  Total size: {total_size / 1e6:.1f} MB")

Semantic model exported: ..\ml-models\semantic
  Embedding dim: 384
  Total size: 91.9 MB


## 2. Re-Embed All Anime

Generate 384-dim embeddings for every anime using the fine-tuned model.
These will be loaded into pgvector for similarity search.

In [4]:
# Load corpus (we embed the full text documents, not just titles)
corpus = []
with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        corpus.append(json.loads(line))

metadata_by_id = {}
metadata_path = DATA_DIR / "anilist_anime.jsonl"
if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            aid = row.get("anilist_id")
            if isinstance(aid, int):
                metadata_by_id[aid] = row
else:
    snapshots_root = DATA_DIR / "catalog_snapshots"
    candidates = [p for p in snapshots_root.glob("catalog_snapshot_*") if (p / "anime_catalog.jsonl").exists()]
    if not candidates:
        raise FileNotFoundError(
            f"Missing {metadata_path} and no catalog snapshots found under {snapshots_root}."
        )
    latest_snapshot = sorted(candidates, key=lambda p: p.name)[-1]
    catalog_path = latest_snapshot / "anime_catalog.jsonl"
    with open(catalog_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            c = json.loads(line)
            aid = c.get("anilist_id")
            if not isinstance(aid, int) or aid <= 0:
                continue
            meta = c.get("metadata_json") if isinstance(c.get("metadata_json"), dict) else {}
            row = {
                "anilist_id": aid,
                "mal_id": c.get("mal_id"),
                "title": c.get("title_english") or c.get("title_romaji") or c.get("title_native") or "",
                "title_romaji": c.get("title_romaji") or meta.get("title_romaji"),
                "title_english": c.get("title_english") or meta.get("title_english"),
                "title_native": c.get("title_native") or meta.get("title_native"),
                "synonyms": meta.get("synonyms") if isinstance(meta.get("synonyms"), list) else [],
                "aliases": meta.get("aliases") if isinstance(meta.get("aliases"), list) else [],
                "genres": c.get("genres") if isinstance(c.get("genres"), list) else (meta.get("genres") if isinstance(meta.get("genres"), list) else []),
                "tags": meta.get("tags") if isinstance(meta.get("tags"), list) else [],
                "format": c.get("format"),
                "status": c.get("status"),
                "season": c.get("season"),
                "season_year": c.get("season_year"),
                "studios": meta.get("studios") if isinstance(meta.get("studios"), list) else [],
                "relations": meta.get("relations") if isinstance(meta.get("relations"), list) else [],
                "description": c.get("description") or meta.get("description"),
                "average_score": c.get("average_score"),
                "popularity": c.get("anilist_popularity") if isinstance(c.get("anilist_popularity"), int) else meta.get("popularity"),
                "anilist_popularity": c.get("anilist_popularity") if isinstance(c.get("anilist_popularity"), int) else meta.get("popularity"),
            }
            metadata_by_id[aid] = row
    print(f"Loaded metadata from snapshot fallback: {latest_snapshot.name}")

print(f"Anime to embed: {len(corpus):,}")
print(f"Metadata rows: {len(metadata_by_id):,}")


Anime to embed: 5,000
Metadata rows: 5,000


In [5]:
# Batch embed all anime
BATCH_SIZE = 64
texts = [entry["text"] for entry in corpus]

print("Embedding all anime...")
all_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

print(f"Embeddings shape: {all_embeddings.shape}")

Embedding all anime...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


In [6]:
# Save embeddings with AniList metadata for backend import/ranking
def _as_text(value):
    if isinstance(value, str):
        text = value.strip()
        return text if text else None
    return None


def _string_list(value):
    out = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                out.append(item.strip())
    elif isinstance(value, str) and value.strip():
        out.append(value.strip())
    return out


def _aliases(meta):
    vals = []
    vals.extend(_string_list(meta.get("synonyms")))
    vals.extend(_string_list(meta.get("aliases")))
    dedup, seen = [], set()
    for item in vals:
        key = item.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(item)
    return dedup


def _tags(meta):
    tags = meta.get("tags")
    if not isinstance(tags, list):
        return []
    out, seen = [], set()
    for tag in tags:
        name = None
        if isinstance(tag, dict):
            name = _as_text(tag.get("name"))
        elif isinstance(tag, str):
            name = _as_text(tag)
        if not name:
            continue
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(name)
    return out


def _studios(meta):
    studios = meta.get("studios")
    if not isinstance(studios, list):
        return []
    out = []
    for studio in studios:
        if isinstance(studio, dict):
            name = _as_text(studio.get("name"))
            if name:
                out.append(name)
        elif isinstance(studio, str) and studio.strip():
            out.append(studio.strip())
    dedup, seen = [], set()
    for name in out:
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(name)
    return dedup


def _relations(meta):
    relations = meta.get("relations")
    if not isinstance(relations, list):
        return []
    out = []
    for rel in relations:
        if isinstance(rel, dict):
            rel_id = rel.get("id")
            if isinstance(rel_id, int):
                out.append(rel_id)
            elif isinstance(rel_id, str) and rel_id.isdigit():
                out.append(int(rel_id))
        elif isinstance(rel, int):
            out.append(rel)
    return out


embeddings_output = []
for i, entry in tqdm(enumerate(corpus), total=len(corpus), desc="Building embedding export"):
    aid = entry["anilist_id"]
    meta = metadata_by_id.get(aid, {})

    title = _as_text(entry.get("title")) or _as_text(meta.get("title")) or ""
    title_romaji = _as_text(meta.get("title_romaji")) or title
    title_english = _as_text(meta.get("title_english"))
    title_native = _as_text(meta.get("title_native"))

    cover_image = meta.get("cover_image")
    if isinstance(cover_image, dict):
        cover_image = (
            _as_text(cover_image.get("extraLarge"))
            or _as_text(cover_image.get("large"))
            or _as_text(cover_image.get("medium"))
        )
    elif not isinstance(cover_image, str):
        cover_image = None

    embeddings_output.append({
        "anilist_id": aid,
        "mal_id": entry.get("mal_id") or meta.get("mal_id"),
        "title": title,
        "title_romaji": title_romaji,
        "title_english": title_english,
        "title_native": title_native,
        "synonyms": _aliases(meta),
        "genres": _string_list(meta.get("genres")),
        "tags": _tags(meta),
        "format": _as_text(meta.get("format")),
        "status": _as_text(meta.get("status")),
        "season": _as_text(meta.get("season")),
        "season_year": meta.get("season_year"),
        "studios": _studios(meta),
        "relations": _relations(meta),
        "description": _as_text(meta.get("description")),
        "cover_image": cover_image,
        "average_score": meta.get("average_score"),
        "anilist_popularity": meta.get("popularity"),
        "embedding_text": entry.get("text"),
        "embedding": all_embeddings[i].tolist(),
    })

timestamped_embeddings = EXPORT_DIR / f"anime_embeddings_{EXPORT_TIMESTAMP}.jsonl"
canonical_embeddings = EXPORT_DIR / "anime_embeddings.jsonl"

with open(timestamped_embeddings, "w", encoding="utf-8") as f:
    for row in tqdm(embeddings_output, desc=f"Writing {timestamped_embeddings.name}"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

shutil.copy2(timestamped_embeddings, canonical_embeddings)

print(f"Saved {len(embeddings_output):,} anime embeddings")
print(f"  Timestamped file: {timestamped_embeddings}")
print(f"  Canonical latest: {canonical_embeddings}")
file_size = timestamped_embeddings.stat().st_size / 1e6
print(f"  Size: {file_size:.1f} MB")


Building embedding export:   0%|          | 0/5000 [00:00<?, ?it/s]

Writing anime_embeddings.jsonl:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved 5,000 anime embeddings
  File: ..\ml-models\anime_embeddings.jsonl
  Size: 60.5 MB


## 3. Export CF Model

In [7]:
# Copy CF checkpoint
cf_src = MODEL_DIR / "anime_cf" / "best_model.pt"
cf_dst = EXPORT_DIR / "cf"
cf_dst.mkdir(exist_ok=True)

shutil.copy2(cf_src, cf_dst / "model.pt")

# Copy anime index mapping
shutil.copy2(DATA_DIR / "cf_anime_index.json", cf_dst / "anime_index.json")

# Copy ID mapping
shutil.copy2(DATA_DIR / "id_map.json", EXPORT_DIR / "id_map.json")

cf_size = (cf_dst / 'model.pt').stat().st_size / 1e6
print(f"CF model exported: {cf_dst}")
print(f"  Checkpoint size: {cf_size:.1f} MB")
print(f"  Anime index: {cf_dst / 'anime_index.json'}")
print(f"  ID mapping: {EXPORT_DIR / 'id_map.json'}")

CF model exported: ..\ml-models\cf
  Checkpoint size: 202.0 MB
  Anime index: ..\ml-models\cf\anime_index.json
  ID mapping: ..\ml-models\id_map.json


## 4. Export Summary

In [8]:
print("=" * 50)
print("EXPORT SUMMARY")
print("=" * 50)

print(f"\nExport directory: {EXPORT_DIR.resolve()}")
print(f"\nContents:")

total_export_size = 0
for f_path in tqdm(sorted(EXPORT_DIR.rglob("*")), desc="Summarizing export"):
    if f_path.is_file():
        size = f_path.stat().st_size
        total_export_size += size
        rel_path = f_path.relative_to(EXPORT_DIR)
        print(f"  {rel_path}: {size / 1e6:.1f} MB")

print(f"\nTotal export size: {total_export_size / 1e6:.1f} MB")

print(f"\n[Deployment Instructions]")
print(f"  1. Download this 'ml-models/' directory")
print(f"  2. Place it at the project root: animetracker/ml-models/")
print(f"  3. Run: docker-compose up --build -d")
print(f"  4. The sidecar will load models from the Docker volume mount")
print(f"\n✓ Export complete!")

EXPORT SUMMARY

Export directory: C:\Users\jeddh\Projects\animetracker\ml-models

Contents:


Summarizing export:   0%|          | 0/23 [00:00<?, ?it/s]

  anime_embeddings.jsonl: 60.5 MB
  anime_embeddings_enriched.jsonl: 44.1 MB
  cf\anime_index.json: 0.6 MB
  cf\item_popularity.json: 0.2 MB
  cf\model.pt: 202.0 MB
  id_map.json: 0.6 MB
  semantic\1_Pooling\config.json: 0.0 MB
  semantic\config.json: 0.0 MB
  semantic\config_sentence_transformers.json: 0.0 MB
  semantic\eval\triplet_evaluation_anime-triplet-eval_results.csv: 0.0 MB
  semantic\model.safetensors: 90.9 MB
  semantic\modules.json: 0.0 MB
  semantic\README.md: 0.0 MB
  semantic\sentence_bert_config.json: 0.0 MB
  semantic\special_tokens_map.json: 0.0 MB
  semantic\tokenizer.json: 0.7 MB
  semantic\tokenizer_config.json: 0.0 MB
  semantic\vocab.txt: 0.2 MB
  semantic_graph.npz: 14.7 MB

Total export size: 414.6 MB

[Deployment Instructions]
  1. Download this 'ml-models/' directory
  2. Place it at the project root: animetracker/ml-models/
  3. Run: docker-compose up --build -d
  4. The sidecar will load models from the Docker volume mount

✓ Export complete!
